## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 1: Content-Based Filtering
## Task 2: User-Profile-Based Content Recommender

Build a content-based recommender that personalizes recommendations based on user preferences.
We construct user profiles from their historical interactions (rated movies) and use them to suggest relevant movies.

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'numpy', '-q'])

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('All imports successful!')

All imports successful!


### Step 1: Load the Dataset

In [2]:
# Download if needed
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_dir = "ml-latest-small"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already exists.")

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))

print(f"Movies: {len(movies)}, Ratings: {len(ratings)}, Users: {ratings['userId'].nunique()}")
movies.head()

Dataset already exists.
Movies: 9742, Ratings: 100836, Users: 610


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### Step 2: Compute TF-IDF Vectors for Movies (from Task 1)

In [3]:
# Prepare genre text and compute TF-IDF
movies['genre_text'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genre_text'] = movies['genre_text'].replace('(no genres listed)', '')

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genre_text'])

# Map movieId -> row index for fast lookup
movie_id_to_idx = pd.Series(movies.index, index=movies['movieId'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Features: {tfidf.get_feature_names_out()}")

TF-IDF matrix shape: (9742, 21)
Features: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'horror' 'imax' 'musical'
 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war' 'western']


### Step 3: Construct User Profiles

For each user $u$:

$$P_u = \frac{\sum_{m \in M} r_{u,m} \cdot f_m}{\sum_{m \in M} r_{u,m}}$$

This is a weighted average of the TF-IDF vectors of movies the user has rated, weighted by their ratings.

In [4]:
def build_user_profiles(ratings_df, tfidf_matrix, movie_id_to_idx):
    """
    Build a user profile vector for each user as a weighted average
    of TF-IDF vectors of their rated movies.
    """
    user_profiles = {}
    
    for user_id, group in ratings_df.groupby('userId'):
        # Filter to movies that exist in our movie index
        valid = group[group['movieId'].isin(movie_id_to_idx.index)]
        if valid.empty:
            continue
        
        # Get TF-IDF vectors for rated movies
        indices = movie_id_to_idx[valid['movieId']].values
        movie_vectors = tfidf_matrix[indices]  # sparse matrix slicing
        user_ratings = valid['rating'].values.reshape(-1, 1)
        
        # Weighted sum of movie vectors
        weighted_sum = (movie_vectors.toarray() * user_ratings).sum(axis=0)
        
        # Normalize by total rating sum
        rating_sum = user_ratings.sum()
        user_profiles[user_id] = weighted_sum / rating_sum
    
    return user_profiles

user_profiles = build_user_profiles(ratings, tfidf_matrix, movie_id_to_idx)
print(f"Built profiles for {len(user_profiles)} users")

# Show example: user 1's genre preferences
example_user = 1
feature_names = tfidf.get_feature_names_out()
profile = user_profiles[example_user]
top_genres = sorted(zip(feature_names, profile), key=lambda x: x[1], reverse=True)[:5]
print(f"\nUser {example_user}'s top genre preferences:")
for genre, weight in top_genres:
    print(f"  {genre}: {weight:.4f}")

Built profiles for 610 users

User 1's top genre preferences:
  adventure: 0.1951
  action: 0.1895
  comedy: 0.1726
  drama: 0.1234
  crime: 0.1228


### Step 4: Compute User-Movie Similarity and Generate Recommendations

$$\text{similarity}(P_u, f_m) = \frac{P_u \cdot f_m}{\|P_u\| \cdot \|f_m\|}$$

In [5]:
def recommend_for_user(user_id, user_profiles, tfidf_matrix, movies_df, ratings_df, top_n=10):
    """
    Recommend top-N movies for a user based on cosine similarity
    between their profile and all movie TF-IDF vectors.
    Excludes movies the user has already rated.
    """
    if user_id not in user_profiles:
        print(f"User {user_id} not found.")
        return None
    
    profile = user_profiles[user_id].reshape(1, -1)
    
    # Cosine similarity between user profile and all movies
    similarities = cosine_similarity(profile, tfidf_matrix).flatten()
    
    # Exclude already-rated movies
    rated_movie_ids = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    rated_indices = set(movie_id_to_idx[mid] for mid in rated_movie_ids if mid in movie_id_to_idx.index)
    
    # Rank all movies, filter out rated ones
    scored = [(i, similarities[i]) for i in range(len(similarities)) if i not in rated_indices]
    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:top_n]
    
    results = pd.DataFrame({
        'Title': [movies_df.iloc[i]['title'] for i, _ in top],
        'Genres': [movies_df.iloc[i]['genres'] for i, _ in top],
        'Similarity': [round(s, 4) for _, s in top]
    })
    return results

### Step 5: Test Recommendations for Sample Users

In [6]:
# Show what user 1 has rated (top-rated movies)
user1_ratings = ratings[ratings['userId'] == 1].merge(movies, on='movieId')
print("User 1's top-rated movies:")
print(user1_ratings.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 Recommendations for User 1")
print("="*70)
recommend_for_user(1, user_profiles, tfidf_matrix, movies, ratings, top_n=10)

User 1's top-rated movies:
                                      title                       genres  rating
               M*A*S*H (a.k.a. MASH) (1970)             Comedy|Drama|War     5.0
                           Excalibur (1981)            Adventure|Fantasy     5.0
  Indiana Jones and the Last Crusade (1989)             Action|Adventure     5.0
                Pink Floyd: The Wall (1982)                Drama|Musical     5.0
               From Russia with Love (1963)    Action|Adventure|Thriller     5.0
                          Goldfinger (1964)    Action|Adventure|Thriller     5.0
                    Dirty Dozen, The (1967)             Action|Drama|War     5.0
                  Gulliver's Travels (1939) Adventure|Animation|Children     5.0
                     American Beauty (1999)                Drama|Romance     5.0
South Park: Bigger, Longer and Uncut (1999)     Animation|Comedy|Musical     5.0

Top 10 Recommendations for User 1


,Title,Genres,Similarity
0,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,0.7910
1,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,0.7736
2,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,0.7731
3,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,0.7731
4,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller,0.7605
5,After the Sunset (2004),Action|Adventure|Comedy|Crime|Thriller,0.7605
6,"Diamond Arm, The (Brilliantovaya ruka) (1968)",Action|Adventure|Comedy|Crime|Thriller,0.7605
7,Machete (2010),Action|Adventure|Comedy|Crime|Thriller,0.7605
8,Maximum Ride (2016),Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller,0.7597
9,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,0.7540


In [7]:
# Recommendations for User 5
user5_ratings = ratings[ratings['userId'] == 5].merge(movies, on='movieId')
print("User 5's top-rated movies:")
print(user5_ratings.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 Recommendations for User 5")
print("="*70)
recommend_for_user(5, user_profiles, tfidf_matrix, movies, ratings, top_n=10)

User 5's top-rated movies:
                                 title                                          genres  rating
             Dances with Wolves (1990)                         Adventure|Drama|Western     5.0
      In the Name of the Father (1993)                                           Drama     5.0
               Schindler's List (1993)                                       Drama|War     5.0
     Postman, The (Postino, Il) (1994)                            Comedy|Drama|Romance     5.0
                      Pinocchio (1940)              Animation|Children|Fantasy|Musical     5.0
           Beauty and the Beast (1991) Animation|Children|Fantasy|Musical|Romance|IMAX     5.0
             Heavenly Creatures (1994)                                     Crime|Drama     5.0
Snow White and the Seven Dwarfs (1937)        Animation|Children|Drama|Fantasy|Musical     5.0
                   Pulp Fiction (1994)                     Comedy|Crime|Drama|Thriller     5.0
             Once Were 

,Title,Genres,Similarity
0,Out of Sight (1998),Comedy|Crime|Drama|Romance|Thriller,0.7868
1,Nurse Betty (2000),Comedy|Crime|Drama|Romance|Thriller,0.7868
2,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,0.7868
3,Band of Outsiders (Bande à part) (1964),Comedy|Crime|Drama|Romance,0.7582
4,"Green Butchers, The (Grønne slagtere, De) (2003)",Comedy|Crime|Drama|Romance,0.7582
5,"Edukators, The (Die Fetten Jahre sind vorbei) ...",Comedy|Crime|Drama|Romance,0.7582
6,Did You Hear About the Morgans? (2009),Comedy|Crime|Drama|Romance,0.7582
7,Focus (2015),Comedy|Crime|Drama|Romance,0.7582
8,Osmosis Jones (2001),Action|Animation|Comedy|Crime|Drama|Romance|Th...,0.7570
9,Freeway (1996),Comedy|Crime|Drama|Thriller,0.7401


In [8]:
# Recommendations for User 100
user100_ratings = ratings[ratings['userId'] == 100].merge(movies, on='movieId')
print("User 100's top-rated movies:")
print(user100_ratings.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 Recommendations for User 100")
print("="*70)
recommend_for_user(100, user_profiles, tfidf_matrix, movies, ratings, top_n=10)

User 100's top-rated movies:
                                                            title                              genres  rating
                                       Terms of Endearment (1983)                        Comedy|Drama     5.0
Christmas Vacation (National Lampoon's Christmas Vacation) (1989)                              Comedy     5.0
                                        Sweet Home Alabama (2002)                      Comedy|Romance     5.0
                                                   Top Gun (1986)                      Action|Romance     5.0
                               Officer and a Gentleman, An (1982)                       Drama|Romance     5.0
                                   When Harry Met Sally... (1989)                      Comedy|Romance     4.5
                                              Out of Sight (1998) Comedy|Crime|Drama|Romance|Thriller     4.5
                                       Wedding Singer, The (1998)                      Come

,Title,Genres,Similarity
0,Waiting to Exhale (1995),Comedy|Drama|Romance,0.9539
1,Mighty Aphrodite (1995),Comedy|Drama|Romance,0.9539
2,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,0.9539
3,Beautiful Girls (1996),Comedy|Drama|Romance,0.9539
4,Something to Talk About (1995),Comedy|Drama|Romance,0.9539
5,Don Juan DeMarco (1995),Comedy|Drama|Romance,0.9539
6,Eat Drink Man Woman (Yin shi nan nu) (1994),Comedy|Drama|Romance,0.9539
7,Nobody's Fool (1994),Comedy|Drama|Romance,0.9539
8,"Corrina, Corrina (1994)",Comedy|Drama|Romance,0.9539
9,I Like It Like That (1994),Comedy|Drama|Romance,0.9539


### Step 6: Evaluate Personalization — Precision@K and Recall@K

To measure recommendation quality, we:
1. Split each user's ratings into a train set (80%) and test set (20%).
2. Build user profiles from the train set.
3. Generate top-K recommendations.
4. Compare recommendations against the test set (movies rated ≥ 4.0 as ground truth).
5. Compute Precision@K and Recall@K.

In [9]:
from sklearn.model_selection import train_test_split

def evaluate_precision_recall(ratings_df, tfidf_matrix, movie_id_to_idx, movies_df, K=10, threshold=4.0):
    """
    Evaluate Precision@K and Recall@K across all users.
    Ground truth: movies in the test set rated >= threshold.
    """
    precisions = []
    recalls = []
    
    for user_id, group in ratings_df.groupby('userId'):
        if len(group) < 5:  # skip users with too few ratings
            continue
        
        # Train/test split
        train, test = train_test_split(group, test_size=0.2, random_state=42)
        
        # Ground truth: highly rated movies in test set
        relevant = set(test[test['rating'] >= threshold]['movieId'])
        if len(relevant) == 0:
            continue
        
        # Build user profile from train set only
        train_profiles = build_user_profiles(train, tfidf_matrix, movie_id_to_idx)
        if user_id not in train_profiles:
            continue
        
        profile = train_profiles[user_id].reshape(1, -1)
        similarities = cosine_similarity(profile, tfidf_matrix).flatten()
        
        # Exclude train-set movies
        train_movie_ids = set(train['movieId'])
        train_indices = set(movie_id_to_idx[mid] for mid in train_movie_ids if mid in movie_id_to_idx.index)
        
        scored = [(i, similarities[i]) for i in range(len(similarities)) if i not in train_indices]
        scored.sort(key=lambda x: x[1], reverse=True)
        top_k_indices = [i for i, _ in scored[:K]]
        
        # Map back to movieIds
        recommended_ids = set(movies_df.iloc[top_k_indices]['movieId'])
        
        # Precision@K and Recall@K
        hits = recommended_ids & relevant
        precisions.append(len(hits) / K)
        recalls.append(len(hits) / len(relevant))
    
    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    return avg_precision, avg_recall, len(precisions)

for K in [5, 10, 20]:
    prec, rec, n_users = evaluate_precision_recall(ratings, tfidf_matrix, movie_id_to_idx, movies, K=K)
    print(f"K={K:>2d} | Precision@K: {prec:.4f} | Recall@K: {rec:.4f} | Evaluated users: {n_users}")

K= 5 | Precision@K: 0.0110 | Recall@K: 0.0057 | Evaluated users: 600
K=10 | Precision@K: 0.0097 | Recall@K: 0.0091 | Evaluated users: 600
K=20 | Precision@K: 0.0098 | Recall@K: 0.0175 | Evaluated users: 600
